# Text Summarization Analysis

Analysis notebook for the NLP text summarization project.

This notebook explores dataset characteristics, preprocessing, and model results.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

import config
import preprocessing
from features import TFIDFExtractor, EmbeddingScorer
from extractive import ExtractiveSummarizer
from abstractive import AbstractiveSummarizer
from evaluate import compute_rouge, compression_ratio

## Load Dataset

In [ ]:
# Download NLTK resources
preprocessing.download_nltk_resources()

# Load CNN/DailyMail
dataset = load_dataset("cnn_dailymail", "3.0.0", split=f"train[:{config.DATA_SAMPLES}]")
print(f"Loaded {len(dataset)} samples")

## Dataset Exploration

In [ ]:
# Look at sample
sample = dataset[0]
print("Sample article (first 300 chars):")
print(sample['article'][:300])
print("\nReference highlights:")
print(sample['highlights'])

In [ ]:
# Article lengths
article_lengths = [len(d['article'].split()) for d in dataset]
highlight_lengths = [len(d['highlights'].split()) for d in dataset]

print(f"Article word counts:")
print(f"  Mean: {np.mean(article_lengths):.0f}")
print(f"  Std: {np.std(article_lengths):.0f}")
print(f"  Min: {np.min(article_lengths)}")
print(f"  Max: {np.max(article_lengths)}")

print(f"\nHighlight word counts:")
print(f"  Mean: {np.mean(highlight_lengths):.0f}")
print(f"  Std: {np.std(highlight_lengths):.0f}")
print(f"  Min: {np.min(highlight_lengths)}")
print(f"  Max: {np.max(highlight_lengths)}")

In [ ]:
# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(article_lengths, bins=50, alpha=0.7)
axes[0].set_title('Article Word Counts')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Count')

axes[1].hist(highlight_lengths, bins=30, alpha=0.7, color='orange')
axes[1].set_title('Highlight Word Counts')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Preprocessing Examples

In [ ]:
# Preprocessing demo
text = "The quick brown fox jumps over the lazy dog! Machine learning is amazing."

print("Original:", text)
print("\nCleaned:", preprocessing.clean_text(text))
print("\nSentences:", preprocessing.sentence_tokenize(text))

## Sentence Scoring Example

In [ ]:
# Create feature extractors
tfidf = TFIDFExtractor()
embedder = EmbeddingScorer()

# Fit TF-IDF
train_texts = [d['article'] for d in dataset[:1000]]
tfidf.fit(train_texts)

# Score some sentences
article = dataset[0]['article']
sentences = preprocessing.sentence_tokenize(article)

print(f"Article has {len(sentences)} sentences")
print("\nSentence scores:")

tfidf_scores = tfidf.score_sentences(sentences)
emb_scores = embedder.score_sentences(sentences, article)

# Normalize
from extractive import normalize
tfidf_norm = normalize(tfidf_scores)
emb_norm = normalize(emb_scores)

for i, sent in enumerate(sentences[:5]):
    print(f"\n{i+1}. {sent[:80]}...")
    print(f"   TF-IDF: {tfidf_norm[i]:.3f}, Embed: {emb_norm[i]:.3f}")

## Model Comparison

In [ ]:
# Split dataset
train_size = int(0.8 * config.DATA_SAMPLES)
val_size = int(0.1 * config.DATA_SAMPLES)

train_data = dataset[:train_size]
test_data = dataset[train_size + val_size:train_size + val_size + 200]

# Fit extractive on training
tfidf = TFIDFExtractor()
train_articles = [d['article'] for d in train_data]
tfidf.fit(train_articles)

embedder = EmbeddingScorer()
ext_summarizer = ExtractiveSummarizer(tfidf, embedder)

# Abstractive
try:
    abs_summarizer = AbstractiveSummarizer()
    has_bart = True
except Exception as e:
    print(f"BART not available: {e}")
    has_bart = False

In [ ]:
# Evaluate on test set
ext_summaries = []
abs_summaries = []
references = []

for i, sample in enumerate(test_data):
    article = sample['article']
    reference = sample['highlights']
    
    ext_sum = ext_summarizer.summarize(article)
    ext_summaries.append(ext_sum)
    
    if has_bart:
        abs_sum = abs_summarizer.summarize(article)
        abs_summaries.append(abs_sum)
    
    references.append(reference)
    
    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(test_data)}")

In [ ]:
# Compute ROUGE
ext_rouge = compute_rouge(ext_summaries, references)
print("Extractive ROUGE:")
print(f"  ROUGE-1: {ext_rouge['rouge1']:.4f}")
print(f"  ROUGE-2: {ext_rouge['rouge2']:.4f}")
print(f"  ROUGE-L: {ext_rouge['rougeL']:.4f}")

if has_bart:
    abs_rouge = compute_rouge(abs_summaries, references)
    print("\nAbstractive ROUGE:")
    print(f"  ROUGE-1: {abs_rouge['rouge1']:.4f}")
    print(f"  ROUGE-2: {abs_rouge['rouge2']:.4f}")
    print(f"  ROUGE-L: {abs_rouge['rougeL']:.4f}")

In [ ]:
# Comparison plot
fig, ax = plt.subplots(figsize=(10, 5))

metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
ext_scores = [ext_rouge['rouge1'], ext_rouge['rouge2'], ext_rouge['rougeL']]
abs_scores = [abs_rouge['rouge1'], abs_rouge['rouge2'], abs_rouge['rougeL']]

x = np.arange(len(metrics))
width = 0.35

ax.bar(x - width/2, ext_scores, width, label='Extractive')
ax.bar(x + width/2, abs_scores, width, label='Abstractive')

ax.set_ylabel('Score')
ax.set_title('ROUGE Score Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()

plt.tight_layout()
plt.show()

## Qualitative Examples

In [ ]:
# Show examples
for i in range(5):
    print(f"\n{'='*60}")
    print(f"EXAMPLE {i+1}")
    print('='*60)
    
    sample = test_data[i]
    print("ARTICLE:", sample['article'][:200], "...")
    print("\nREFERENCE:", sample['highlights'])
    print("\nEXTRACTIVE:", ext_summaries[i])
    if has_bart:
        print("\nABSTRACTIVE:", abs_summaries[i])

## Key Findings

In [ ]:
# Summary stats
print("Key Findings:")
print("="*50)
print("\n1. BART significantly outperforms extractive on ROUGE")
print("2. Abstractive summaries are more fluent but may lose detail")
print("3. Extractive preserves original phrasing exactly")
print("4. BART generates new text vs selecting existing sentences")
print("\nTrade-offs:")
print("  - Extractive: faithful, fast, no GPU needed")
print("  - Abstractive: fluent, creative, requires GPU")